In [ ]:
!pip install darts -q


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
from pathlib import Path

from darts import TimeSeries
from sklearn.preprocessing import MinMaxScaler
from darts.models import RNNModel
from darts.metrics import rmse, mae, mape
from pytorch_lightning.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import numpy as np
import torch
torch.set_float32_matmul_precision("high")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- CONFIG ---
# Path to Subnet Results
SUBNET_PATH = "/content/drive/MyDrive/Thesis/New_Results/Subnets/GRU_1layer_log/gru_1layer_subnets_global_rolling_metrics.csv"

# Path to Institution Results
INSTITUTION_PATH = "/content/drive/MyDrive/Thesis/New_Results/Institutions/GRU_1layer_global/gru_1layer_institution_rolling_metrics.csv"
# --- LOAD DATA ---
try:
    df_subnet = pd.read_csv(SUBNET_PATH)
    df_inst = pd.read_csv(INSTITUTION_PATH)

    # Standardize column names if needed
    if 'r2' in df_inst.columns:
        df_inst.rename(columns={'r2': 'R2_LogScaled'}, inplace=True)

    # --- CALCULATE STATS ---
    subnet_median = df_subnet['R2_LogScaled'].median()
    inst_median = df_inst['R2_LogScaled'].median()

    # Calculate "Win Rate" (How many institutions have R2 > 0.5 compared to subnets?)
    subnet_good = (len(df_subnet[df_subnet['R2_LogScaled'] > 0.5]) / len(df_subnet)) * 100
    inst_good = (len(df_inst[df_inst['R2_LogScaled'] > 0.5]) / len(df_inst)) * 100

    print("=== RQ3 GRANULARITY RESULTS ===")
    print(f"Subnet Median R2:      {subnet_median:.4f} (N={len(df_subnet)})")
    print(f"Institution Median R2: {inst_median:.4f} (N={len(df_inst)})")
    print(f"Subnets > 0.5 R2:      {subnet_good:.1f}%")
    print(f"Institutions > 0.5 R2: {inst_good:.1f}%")

    # --- PLOT ---
    df_subnet['Level'] = 'Subnet (Fine-Grained)'
    df_inst['Level'] = 'Institution (Aggregated)'

    # Select only the R2 column and Level
    plot_data = pd.concat([
        df_subnet[['R2_LogScaled', 'Level']],
        df_inst[['R2_LogScaled', 'Level']]
    ])

    # Clip outliers
    plot_data['R2_LogScaled'] = plot_data['R2_LogScaled'].clip(lower=-1.0)

    plt.figure(figsize=(10, 6))
    sns.boxplot(x='Level', y='R2_LogScaled', data=plot_data, palette="Blues", width=0.5)

    plt.title("Impact of Granularity: Subnet vs. Institution Level", fontsize=14)
    plt.ylabel("Coefficient of Determination ($R^2$)", fontsize=12)
    plt.xlabel("")
    plt.grid(True, axis='y', alpha=0.3)
    plt.axhline(0, color='red', linestyle='--', linewidth=1, label="Baseline")

    plt.tight_layout()
    plt.savefig("granularity_comparison.png")
    plt.show()
    print("Saved plot as 'granularity_comparison.png'")

except Exception as e:
    print(f"Error: {e}")
    print("Please check the file path for the Institution Results CSV.")

In [ ]:
import pandas as pd

# --- CONFIG ---
GLOBAL_PATH = "/content/drive/MyDrive/Thesis/New_Results/Subnets/GRU_1layer_log/gru_1layer_subnets_global_rolling_metrics.csv"
CLUSTERED_PATH = "/content/drive/MyDrive/Thesis/New_Results/Subnets/Clustering/GRU_1L_log/Cluster_Results_Rolling/gru_subnets_1log_cluster_metrics.csv"

# --- LOAD ---
df_global = pd.read_csv(GLOBAL_PATH)
df_cluster = pd.read_csv(CLUSTERED_PATH)

# Clean
df_global['subnet'] = df_global['subnet'].astype(str)
df_cluster['subnet'] = df_cluster['subnet'].astype(str)
df_cluster.rename(columns={'R2_Scaled': 'R2_LogScaled'}, inplace=True)

# Merge
merged = pd.merge(df_global[['subnet', 'R2_LogScaled']],
                  df_cluster[['subnet', 'R2_LogScaled']],
                  on='subnet', suffixes=('_Global', '_Clustered'))

# --- CALCULATE METRICS ---
merged['Global_Wins'] = merged['R2_LogScaled_Global'] > merged['R2_LogScaled_Clustered']
global_wins = merged['Global_Wins'].sum()
total = len(merged)
win_rate = (global_wins / total) * 100

# 2. Stability
# We define "Failure" as R2 < 0 (Worse than mean)
global_failures = len(merged[merged['R2_LogScaled_Global'] < 0])
cluster_failures = len(merged[merged['R2_LogScaled_Clustered'] < 0])

# 3. Catastrophic Failures (R2 < -10)
global_crash = len(merged[merged['R2_LogScaled_Global'] < -10])
cluster_crash = len(merged[merged['R2_LogScaled_Clustered'] < -10])

print("=== HEAD-TO-HEAD RESULTS ===")
print(f"Total Subnets Compared: {total}")
print(f"Global Model Wins:      {global_wins} ({win_rate:.1f}%)")
print(f"Clustered Model Wins:   {total - global_wins} ({100 - win_rate:.1f}%)")
print("\n=== STABILITY ANALYSIS ===")
print(f"Global Failures (R2 < 0):   {global_failures}")
print(f"Clustered Failures (R2 < 0):{cluster_failures}")
print(f"Global Crashes (R2 < -10):  {global_crash}")
print(f"Clustered Crashes (R2<-10): {cluster_crash}")

In [ ]:
import pandas as pd

# --- CONFIG ---
CLUSTERS_PATH ="/content/drive/MyDrive/Thesis/Datasets/Clustering/subnet_clusters_raw.csv"

# Load Data
df = pd.read_csv(CLUSTERS_PATH)

# Calculate Statistics per Cluster
# We use:
# - bytes_mean: How big is the traffic?
# - bytes_cv: How "spiky" is it? (Coefficient of Variation)
# - bytes_zero_frac: How often is it dead?
stats = df.groupby('cluster')[['bytes_mean', 'bytes_cv', 'bytes_zero_frac']].mean()
counts = df['cluster'].value_counts()

print("\n=== CLUSTER STATISTICS (For LaTeX Table) ===")
print(f"Cluster 0 Count: {counts.get(0, 0)}")
print(f"Cluster 1 Count: {counts.get(1, 0)}")
print("\nAverage Values:")
print(stats)

In [ ]:
DATA_DIR   = Path("/content/drive/MyDrive/Thesis/Datasets/network_anomaly")
SUBNET_DIR = DATA_DIR / "institution_subnets/agg_1_hour"

TIMES_PATH = DATA_DIR / "times/times_1_hour.csv"
FEATURE = "n_bytes"
FEATURE_COLS = [FEATURE]

times_df = pd.read_csv(TIMES_PATH)

times_df["time"] = pd.to_datetime(times_df["time"], errors="coerce")
if times_df["time"].dt.tz is not None:
    times_df["time"] = times_df["time"].dt.tz_localize(None)

times_df["id_time"] = times_df["id_time"].astype(int)

from darts import TimeSeries

raw_series = {}
ts_dict    = {}

for csv_path in SUBNET_DIR.glob("*.csv"):
    sid = csv_path.stem
    df = pd.read_csv(csv_path)

    df["id_time"] = df["id_time"].astype(int)

    df = df.merge(times_df, on="id_time", how="left")

    df = df.sort_values("time")
    df = df.set_index("time")

    #keep only n_bytes
    df = df[FEATURE_COLS].copy()

    raw_series[sid] = df

    ts = TimeSeries.from_dataframe(
        df,
        value_cols=FEATURE_COLS,  # this is just ["n_bytes"]
        fill_missing_dates=True,
        freq="h"
    )
    ts_dict[sid] = ts

print(f"Loaded {len(ts_dict)} subnets into ts_dict")



In [ ]:
from darts.dataprocessing.transformers import MissingValuesFiller
import numpy as np

filler = MissingValuesFiller()

filled_ts_dict = {}

for sid, ts in ts_dict.items():
    ts_filled = filler.transform(ts)
    n_nans = np.isnan(ts_filled.values()).sum()
    print(f"{sid}: NaNs AFTER filling = {n_nans}")
    filled_ts_dict[sid] = ts_filled


In [ ]:
train_ts_dict = {}
val_ts_dict   = {}
test_ts_dict  = {}

for sid, ts in filled_ts_dict.items():
    n = len(ts)
    train_end = int(n * 0.35)
    val_end   = int(n * 0.40)  # 35% + 5%

    train_ts_dict[sid] = ts[:train_end]
    val_ts_dict[sid]   = ts[train_end:val_end]
    test_ts_dict[sid]  = ts[val_end:]


In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

train_scaled_log = {}
val_scaled_log   = {}
test_scaled_log  = {}
scalers_log      = {}   # per-subnet scaler in log-space

for sid in train_ts_dict.keys():
    train_ts = train_ts_dict[sid]
    val_ts   = val_ts_dict[sid]
    test_ts  = test_ts_dict[sid]

    # raw values (n_bytes)
    train_vals = train_ts.values()   # (T_train, 1)
    val_vals   = val_ts.values()
    test_vals  = test_ts.values()

    # ---- log1p transform ----
    train_log = np.log1p(train_vals)
    val_log   = np.log1p(val_vals)
    test_log  = np.log1p(test_vals)

    # concatenate train+val in log-space
    tv_log = np.concatenate([train_log, val_log], axis=0)

    scaler = MinMaxScaler(feature_range=(0, 1))
    scaler.fit(tv_log)

    train_scaled_vals = scaler.transform(train_log)
    val_scaled_vals   = scaler.transform(val_log)
    test_scaled_vals  = scaler.transform(test_log)

    # wrap back into TimeSeries
    train_scaled_log[sid] = train_ts.with_values(train_scaled_vals)
    val_scaled_log[sid]   = val_ts.with_values(val_scaled_vals)
    test_scaled_log[sid]  = test_ts.with_values(test_scaled_vals)

    scalers_log[sid] = scaler


train_scaled = {}
val_scaled   = {}
test_scaled  = {}
scalers      = {}   # subnet_id -> its own MinMaxScaler

for sid in train_ts_dict.keys():
    train_ts = train_ts_dict[sid]
    val_ts   = val_ts_dict[sid]
    test_ts  = test_ts_dict[sid]

    # concatenate train+val like the paper's X_train_val
    tv_vals = np.concatenate([train_ts.values(), val_ts.values()], axis=0)  # shape (N,1)

    scaler = MinMaxScaler(feature_range=(0, 1))
    scaler.fit(tv_vals)

    # scale each split separately using the SAME scaler
    train_scaled_vals = scaler.transform(train_ts.values())
    val_scaled_vals   = scaler.transform(val_ts.values())
    test_scaled_vals  = scaler.transform(test_ts.values())

    train_scaled[sid] = train_ts.with_values(train_scaled_vals)
    val_scaled[sid]   = val_ts.with_values(val_scaled_vals)
    test_scaled[sid]  = test_ts.with_values(test_scaled_vals)

    scalers[sid] = scaler
valid_ids = []

In [ ]:
from pathlib import Path
import pandas as pd

BASE_OUT = Path("/content/drive/MyDrive/Thesis/New_Results/Clustering")
CLUSTERS= Path("/content/drive/MyDrive/Thesis/Datasets/Clustering/")
CLUSTERS_PATH = CLUSTERS / "subnet_clusters_raw.csv"

clusters_df = pd.read_csv(CLUSTERS_PATH, index_col="subnet")

clusters_df.index = clusters_df.index.astype(str)

# Also make your scaled dicts keyed by str
train_scaled_log = {str(sid): ts for sid, ts in train_scaled_log.items()}
val_scaled_log   = {str(sid): ts for sid, ts in val_scaled_log.items()}
test_scaled_log  = {str(sid): ts for sid, ts in test_scaled_log.items()}
scalers_log      = {str(sid): sc for sid, sc in scalers_log.items()}

train_scaled = {str(sid): ts for sid, ts in train_scaled.items()}
val_scaled   = {str(sid): ts for sid, ts in val_scaled.items()}
test_scaled  = {str(sid): ts for sid, ts in test_scaled.items()}
scalers      = {str(sid): sc for sid, sc in scalers.items()}

cluster_map = clusters_df["cluster"].to_dict()
cluster_ids = sorted(clusters_df["cluster"].unique())

print("Clusters:", cluster_ids)
print("Cluster sizes:\n", clusters_df["cluster"].value_counts())


In [ ]:
INPUT_LEN  = 168
STEP       = 1

def get_cluster_subnets(cluster_id, cluster_map, available_sids):
    return [
        sid for sid, cl in cluster_map.items()
        if cl == cluster_id and sid in available_sids
    ]

all_sids_scaled = set(train_scaled_log.keys())


In [ ]:
from pathlib import Path

INPUT_CHUNK_LENGTH = 168
MIN_LEN = INPUT_CHUNK_LENGTH + 1

cluster_models = {}
cluster_valid_ids = {}

for c in cluster_ids:
    print(f"\n=== Preparing data for GRU_1L_log cluster {c} ===")

    subnets_in_c = [
        sid for sid, cl in cluster_map.items()
        if cl == c and sid in train_scaled
    ]

    train_list_c = []
    val_list_c   = []
    valid_ids_c  = []

    for sid in subnets_in_c:
        len_train = len(train_scaled[sid])
        len_val   = len(val_scaled[sid])

        if len_train >= MIN_LEN and len_val >= MIN_LEN:
            valid_ids_c.append(sid)
            train_list_c.append(train_scaled[sid])
            val_list_c.append(val_scaled[sid])
        else:
            print(f"  Dropping {sid} in cluster {c}: "
                  f"len_train={len_train}, len_val={len_val}")

    if not valid_ids_c:
        print(f"  -> No usable subnets for cluster {c}, skipping.")
        continue

    print(f"  -> {len(valid_ids_c)} valid subnets in cluster {c}")
    cluster_valid_ids[c] = valid_ids_c


In [ ]:
from darts.models import RNNModel


MODEL_DIR = Path("/content/drive/MyDrive/Thesis/New_Results/Subnets/Clustering/GRU_1L_log")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

cluster_models = {}

for c, valid_ids_c in cluster_valid_ids.items():
    print(f"\n=== Training GRU_1L for cluster {c} ===")
    print(f"Subnets in cluster {c}: {len(valid_ids_c)}")

    # Build the series lists for this cluster
    train_list_c = [train_scaled_log[sid] for sid in valid_ids_c]
    val_list_c   = [val_scaled_log[sid]   for sid in valid_ids_c]


    early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    min_delta=1e-4,
    mode="min",
    )

    gru_cluster = RNNModel(
        model="GRU",
        input_chunk_length=INPUT_CHUNK_LENGTH,
        training_length=INPUT_CHUNK_LENGTH,
        hidden_dim=64,
        n_rnn_layers=1,
        dropout=0.1,
        batch_size=64,
        n_epochs=30,
        optimizer_kwargs={"lr": 1e-3},
        random_state=42,
        model_name=f"gru_subnets_1log_cluster_{c}",
        pl_trainer_kwargs={
            "accelerator": "gpu",
            "devices": 1,
            "gradient_clip_val": 1.0,
            "callbacks": [early_stop],  # same callback you used
        },
    )

    gru_cluster.fit(
        series=train_list_c,
        val_series=val_list_c,
        verbose=True,
    )


    # Save model
    ckpt_path = MODEL_DIR / f"gru_subnets_1log_cluster_{c}.pth.tar"
    gru_cluster.save(str(ckpt_path))
    print("  -> Saved cluster GRU to:", ckpt_path)

    cluster_models[c] = gru_cluster


In [ ]:
from darts.models import RNNModel
from darts.metrics import rmse, r2_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# 1. Define Paths
MODEL_DIR = Path("/content/drive/MyDrive/Thesis/New_Results/Subnets/Clustering/GRU_1L_log")

# Directory to save results
CLUSTER_RES_DIR = MODEL_DIR / "Cluster_Results_Rolling"
CLUSTER_RES_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR = CLUSTER_RES_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

all_cluster_results = []

print(f"Starting CLUSTERED ROLLING evaluation on {len(cluster_valid_ids)} clusters...")

# 2. Loop through each cluster
for c, valid_ids_c in cluster_valid_ids.items():
    print(f"\n=== Processing Cluster {c} ({len(valid_ids_c)} subnets) ===")

    # --- LOAD MODEL FROM DISK ---
    model_name = f"gru_subnets_1log_cluster_{c}.pth.tar"
    load_path = MODEL_DIR / model_name

    try:
        current_model = RNNModel.load(str(load_path))
        print(f"  ✅ Loaded model: {model_name}")
    except Exception as e:
        print(f"Could not load {model_name}. Skipping cluster {c}. Error: {e}")
        continue

    # --- EVALUATE SUBNETS IN THIS CLUSTER ---
    for i, sid in enumerate(valid_ids_c):

        # Ensure ID is string to match dictionary keys
        sid_str = str(sid)

        # Get Log-Scaled Data
        train_ts = train_scaled[sid_str]
        val_ts   = val_scaled[sid_str]
        test_ts  = test_scaled[sid_str]

        # Concatenate history
        full_series = train_ts.concatenate(val_ts).concatenate(test_ts)

        # 3. ROLLING FORECAST
        #    Predict 1 step ahead, sliding forward 1 step at a time
        try:
            pred = current_model.historical_forecasts(
                series=full_series,
                start=test_ts.start_time(),
                forecast_horizon=1,
                stride=1,
                retrain=False,
                verbose=False,
                last_points_only=True
            )

            # 4. Metrics
            r2_val = r2_score(test_ts, pred)
            rmse_val = rmse(test_ts, pred)

            all_cluster_results.append({
                "subnet": sid_str,
                "cluster": c,
                "R2_Scaled": r2_val,
                "RMSE_Scaled": rmse_val,
                "n_test_points": len(test_ts)
            })

            # 5. Plotting (Sample first 2 subnets per cluster)
            if i < 2:
                plt.figure(figsize=(12, 5))
                plt.plot(test_ts.time_index, test_ts.values(), label="Actual (Log)", color='blue', alpha=0.5)
                plt.plot(pred.time_index, pred.values(), label=f"Cluster {c} Model", color='orange', alpha=0.8)

                plt.title(f"Cluster {c} | Subnet {sid_str} | R2: {r2_val:.4f}")
                plt.legend()
                plt.grid(True, alpha=0.3)

                p_file = PLOT_DIR / f"cluster_{c}_subnet_{sid_str}.png"
                plt.savefig(p_file)
                plt.close()
                print(f"   -> Plot saved for subnet {sid_str}")

        except Exception as e:
            print(f"Error evaluating subnet {sid_str}: {e}")

# --- SAVE MASTER CSV ---
if all_cluster_results:
    df_cluster_res = pd.DataFrame(all_cluster_results)

    print("\n=== Cluster Evaluation Summary ===")
    print(df_cluster_res.groupby("cluster")[["R2_Scaled", "RMSE_Scaled"]].mean())

    FINAL_CSV = CLUSTER_RES_DIR / "gru_subnets_1log_cluster_metrics.csv"
    df_cluster_res.to_csv(FINAL_CSV, index=False)
    print(f"\nAll cluster results saved to: {FINAL_CSV}")
else:
    print("\nNo results were generated. Check your paths/data.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- CONFIG: UPDATE PATHS ---
GLOBAL_PATH = "/content/drive/MyDrive/Thesis/New_Results/Subnets/GRU_1layer_log/gru_1layer_subnets_global_rolling_metrics.csv"

CLUSTERED_PATH = "/content/drive/MyDrive/Thesis/New_Results/Subnets/Clustering/GRU_1L_log/Cluster_Results_Rolling/gru_subnets_1log_cluster_metrics.csv"

# --- LOAD DATA ---
df_global = pd.read_csv(GLOBAL_PATH)
df_cluster = pd.read_csv(CLUSTERED_PATH)

# --- CLEANING & MERGING ---
# 1. Ensure Subnet IDs are strings for accurate matching
df_global['subnet'] = df_global['subnet'].astype(str)
df_cluster['subnet'] = df_cluster['subnet'].astype(str)

# 2. Rename columns in Clustered DF to match Global DF convention
df_cluster.rename(columns={'R2_Scaled': 'R2_LogScaled', 'RMSE_Scaled': 'RMSE_LogScaled'}, inplace=True)

# 3. Merge the two DataFrames on 'subnet'
merged = pd.merge(
    df_global[['subnet', 'R2_LogScaled', 'RMSE_LogScaled']],
    df_cluster[['subnet', 'cluster', 'R2_LogScaled', 'RMSE_LogScaled']],
    on='subnet',
    suffixes=('_Global', '_Clustered')
)

# --- ANALYSIS ---
# Calculate the "Clustering Lift" (How much did clustering help?)
# Positive Value = Clustered Model is Better
# Negative Value = Global Model is Better
merged['R2_Diff'] = merged['R2_LogScaled_Clustered'] - merged['R2_LogScaled_Global']

# Group by Cluster to see the average impact per behavior type
summary = merged.groupby('cluster').agg({
    'R2_LogScaled_Global': 'mean',
    'R2_LogScaled_Clustered': 'mean',
    'R2_Diff': 'mean',
    'subnet': 'count'
}).rename(columns={'subnet': 'count'})

print("\n=== SUMMARY TABLE: Global vs. Clustered Performance ===")
print(summary)

# --- DETAILED PRINTOUT ---
print("\n=== DETAILED FINDINGS ===")
for cluster_id in summary.index:
    diff = summary.loc[cluster_id, 'R2_Diff']
    winner = "CLUSTERED" if diff > 0 else "GLOBAL"
    print(f"Cluster {cluster_id}: {winner} won (Avg Diff: {diff:.4f})")

# --- VISUALIZATION ---
plot_data = merged.melt(
    id_vars=['subnet', 'cluster'],
    value_vars=['R2_LogScaled_Global', 'R2_LogScaled_Clustered'],
    var_name='Strategy',
    value_name='R2 Score'
)

plot_data['Strategy'] = plot_data['Strategy'].replace({
    'R2_LogScaled_Global': 'Global Model (Generalist)',
    'R2_LogScaled_Clustered': 'Cluster Model (Specialist)'
})

plt.figure(figsize=(10, 6))
sns.barplot(
    data=plot_data,
    x='cluster',
    y='R2 Score',
    hue='Strategy',
    palette='viridis',
    errorbar=None
)

plt.title("RQ2 Results: Does Specialization Improve Accuracy?", fontsize=14, fontweight='bold')
plt.ylabel("Average $R^2$ Score", fontsize=12)
plt.xlabel("Traffic Behavior Cluster", fontsize=12)
plt.legend(title=None)
plt.grid(True, axis='y', alpha=0.3)
plt.axhline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.savefig("global_vs_clustered_barplot.png")
plt.show()
print("Saved comparison plot as 'global_vs_clustered_barplot.png'")

In [ ]:
# Check exact scores for Subnet 101
sid = '101'
row_global = df_global[df_global['subnet'] == sid]
row_cluster = df_cluster[df_cluster['subnet'] == sid]

print(f"--- Subnet {sid} ---")
print(f"Global R2:    {row_global['R2_LogScaled'].values[0]:.4f}")
print(f"Clustered R2: {row_cluster['R2_LogScaled'].values[0]:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from darts.models import RNNModel
from pathlib import Path

# --- CONFIG ---
MODEL_PATH = "/content/drive/MyDrive/Thesis/New_Results/Subnets/GRU_1layer_log/gru_168_1layer_subnets_log.pth.tar"

# --- LOAD MODEL ---
print("Loading Global Model...")
model = RNNModel.load(str(MODEL_PATH))

# ---BIGGEST ANOMALY ---
max_error = 0
best_subnet = None
best_pred = None
best_actual = None

print("Scanning subnets for anomalies...")

for sid in list(test_scaled_log.keys())[:100]:
    try:
        train = train_scaled_log[sid]
        val = val_scaled_log[sid]
        test = test_scaled_log[sid]

        # We only care about the Test set
        # Predict the next 168 hours (1 week) starting from end of Val
        input_series = train.concatenate(val)

        # Predict
        pred = model.predict(n=168, series=input_series, verbose=False)

        # Calculate Error (Residuals)
        # We need to slice 'test' to match the prediction length
        actual_slice = test[:168]

        # Calculate max absolute difference
        diff = np.abs(actual_slice.values() - pred.values())
        current_max = np.max(diff)

        if current_max > max_error:
            max_error = current_max
            best_subnet = sid
            best_pred = pred
            best_actual = actual_slice

    except Exception as e:
        continue

print(f"Found biggest anomaly in Subnet: {best_subnet} (Max Error: {max_error:.4f})")

# --- PLOT THE DETECTED ANOMALY ---
plt.figure(figsize=(12, 6))

# Plot Actual (Blue)
best_actual.plot(label="Actual Traffic (Anomaly)", color='#e74c3c', linewidth=2)

# Plot Prediction (Green)
best_pred.plot(label="Predicted Normal Behavior", color='#27ae60', linewidth=2, linestyle='--')

# Highlight the Gap
plt.fill_between(best_actual.time_index,
                 best_actual.values().flatten(),
                 best_pred.values().flatten(),
                 color='gray', alpha=0.3, label="Anomaly Magnitude (Error)")

plt.title(f"Anomaly Detection Case Study: Subnet {best_subnet}", fontsize=14, fontweight='bold')
plt.ylabel("Log-Scaled Traffic Volume")
plt.xlabel("Time")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("anomaly_detection_example.png")
plt.show()
print("Saved plot as 'anomaly_detection_example.png'")